In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
transcribe=pd.read_csv('/content/merged_captioning_transcriptions.csv')
caption_id=pd.read_csv('/content/captioning_translated_nllb.csv')

In [ ]:
transcribe.head()

,path,caption,base_name,id,file,emotion,text
0,Trust/1.mp4,"['a woman and a child sitting on a chair\n', '...",1,NaN,1.wav,Trust,5 tenbe ban di rumah untuk lansia yang pertam...
1,Trust/3.mp4,['a woman in red shirt is running on a red and...,3,NaN,3.wav,Trust,Akhirnya gue ikut event lari lagi setelah ski...
2,Trust/11.mp4,['a man and woman sitting on a bench in a gym\...,11,NaN,11.wav,Trust,"Gak ada aturan pastinya, mesti kemana. Mau ke..."
3,Trust/12.mp4,"['a woman is using a machine in a gym\n', 'a w...",12,NaN,12.wav,Trust,"gym iya, trail run iya, yoga iya, tennis iya,..."
4,Trust/15.mp4,"['in cara bentuk - ini cara bentuk\n', 'a man ...",15,NaN,15.wav,Trust,"Ini cara bentuk balanmu di rumah tanpa alat, ..."


In [ ]:
import kagglehub
path_kamus_1= kagglehub.dataset_download("fornigulo/kamus-slag")
path_kamus_2= pd.read_csv('/content/colloquial-indonesian-lexicon.csv')
path_kamus_2['tidak_baku']=path_kamus_2['slang'].values
path_kamus_2['kata_baku']=path_kamus_2['formal'].values
kamus_normalisasi = pd.read_excel(path_kamus_1 + '/kamuskatabaku.xlsx')
kamus_normalisasi=pd.concat([kamus_normalisasi,path_kamus_2[['tidak_baku','kata_baku']]])
kamus_path_3=pd.read_csv('slang.csv')
kamus_path_3=kamus_path_3.rename(columns={
    'slang':'tidak_baku',
    'formal':'kata_baku'
})
kamus_normalisasi=pd.concat([kamus_normalisasi,kamus_path_3])
kamus_normalisasi.drop_duplicates(inplace=True)

Using Colab cache for faster access to the 'kamus-slag' dataset.


In [ ]:
kamus_dict = dict(zip(kamus_normalisasi['tidak_baku'].str.strip(), kamus_normalisasi['kata_baku'].str.strip()))

# Contoh fungsi normalisasi token
def normalisasi_dengan_kamus(tokens):
    return [kamus_dict.get(t, t) for t in tokens]

In [ ]:
transcribe['text_normalisasi'] = transcribe['text'].astype(str).str.lower().apply(lambda x:x.split()).apply(normalisasi_dengan_kamus).apply(lambda x: ' '.join(x))

In [ ]:
transcribe['text_normalisasi'].sample(10)

,text_normalisasi
879,"kamu bandingkan sama tv dah, di lepehin 11 inc..."
423,kalau kalian hanya ada 30 menit lakukan latiha...
619,hari ini ada jadwal apa ya? kosong apa yang si...
904,"hai, aku amanda ralls dan sekarang aku lagi ad..."
537,di atas semua mobil lecet-lecet semua tapi str...
21,"sebenarnya ini ya, kalian melakukan rose atau ..."
636,ini yang perlu kamu tahu untuk jadi pemain bas...
198,jadi di perjalanan satu million journey edisi ...
941,"3 gerakan basic dalam paddle pertama, forehand..."
837,banyak sekali yang bertanya sama aku kalau dok...


In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

# Get Indonesian stopwords from NLTK
stopwords_indonesia = set(stopwords.words('indonesian'))

# Create a custom stopwords list
custom_stopwords = set(['e', 'eh','em','ee','eee']) # Add your custom stopwords here

# Combine NLTK stopwords and custom stopwords
all_stopwords = stopwords_indonesia.union(custom_stopwords)

# Function to remove stopwords
def remove_stopwords(tokens):
    return [t for t in tokens if t not in all_stopwords]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
transcribe['text_without_stopwords'] = transcribe['text_normalisasi'].apply(lambda x: x.split()).apply(remove_stopwords).apply(lambda x: ' '.join(x))
display(transcribe[['text_normalisasi', 'text_without_stopwords']].head())

,text_normalisasi,text_without_stopwords
0,5 tenbe ban di rumah untuk lansia yang pertama...,"5 tenbe ban rumah lansia pertama, seat stand g..."
1,akhirnya aku ikut acara lari lagi setelah skip...,acara lari skip 1 bandung bangun pagi buta aca...
2,"enggak ada aturan pastinya, mesti kemana. mau ...","aturan pastinya, mesti kemana. dokter umum, do..."
3,"gym iya, trail run iya, yoga iya, tennis iya, ...","gym iya, trail run iya, yoga iya, tennis iya, ..."
4,"ini cara bentuk balanmu di rumah tanpa alat, u...","bentuk balanmu rumah alat, unitonoris. gerakan..."


In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 18.5 MB/s eta 0:00:00


In [ ]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Function to remove punctuation
def remove_punctuation(text):
    text = re.sub(r'[^\w\s]', '', text)
    return text

# Initialize Sastrawi stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Function to perform stemming (lemmatization)
def stem_text(text):
    return stemmer.stem(text)

In [ ]:
transcribe['text_without_punctuation'] = transcribe['text_without_stopwords'].apply(remove_punctuation)
transcribe['text_stemmed'] = transcribe['text_without_punctuation'].apply(stem_text)

display(transcribe[['text_without_stopwords', 'text_without_punctuation', 'text_stemmed']].head())

,text_without_stopwords,text_without_punctuation,text_stemmed
0,"5 tenbe ban rumah lansia pertama, seat stand g...",5 tenbe ban rumah lansia pertama seat stand ge...,5 tenbe ban rumah lansia pertama seat stand ge...
1,acara lari skip 1 bandung bangun pagi buta aca...,acara lari skip 1 bandung bangun pagi buta aca...,acara lari skip 1 bandung bangun pagi buta aca...
2,"aturan pastinya, mesti kemana. dokter umum, do...",aturan pastinya mesti kemana dokter umum dokte...,atur pasti mesti mana dokter umum dokter olahr...
3,"gym iya, trail run iya, yoga iya, tennis iya, ...",gym iya trail run iya yoga iya tennis iya pila...,gym iya trail run iya yoga iya tennis iya pila...
4,"bentuk balanmu rumah alat, unitonoris. gerakan...",bentuk balanmu rumah alat unitonoris gerakan l...,bentuk balan rumah alat unitonoris gera lunge ...


In [ ]:
caption_id.head()

,path,caption,caption_id
0,Trust/1.mp4,"['a woman and a child sitting on a chair\n', '...",['seorang wanita dan seorang anak duduk di kur...
1,Trust/3.mp4,['a woman in red shirt is running on a red and...,['seorang wanita dengan baju merah sedang berl...
2,Trust/11.mp4,['a man and woman sitting on a bench in a gym\...,['seorang pria dan wanita duduk di bangku di g...
3,Trust/12.mp4,"['a woman is using a machine in a gym\n', 'a w...","['seorang wanita menggunakan mesin di gym', 's..."
4,Trust/15.mp4,"['in cara bentuk - ini cara bentuk\n', 'a man ...","['in cara bentuk - ini cara bentuk ', 'Seorang..."


In [ ]:
transcribe['caption_id']=caption_id['caption_id'].apply(lambda x: ' '.join(eval(x)) if isinstance(x, str) else ' '.join(x))

In [ ]:
transcribe['caption_id']

,caption_id
0,seorang wanita dan seorang anak duduk di kursi...
1,seorang wanita dengan baju merah sedang berlar...
2,seorang pria dan wanita duduk di bangku di gym...
3,seorang wanita menggunakan mesin di gym seoran...
4,in cara bentuk - ini cara bentuk Seorang pria...
...,...
993,logo seo ditampilkan di latar belakang biru S...
994,seorang wanita hamil duduk di tempat tidur den...
995,seorang wanita dengan jaket putih tersenyum. s...
996,seseorang mengenakan celana jeans dan kaos hit...


In [ ]:
transcribe['text_stemed+caption']=transcribe['text_stemmed']+' '+transcribe['caption_id']

In [ ]:
!pip install bertopic top2vec contextualized-topic-models sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.1 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Unins

In [ ]:
import bertopic
print(bertopic.__version__)


0.17.3


In [ ]:
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# ------------------------------------------------------
# 1️ Load data multimodal (text + caption)
# ------------------------------------------------------
transcribe = pd.read_csv('/content/transcribe_bersih.csv')
texts = transcribe['text_stemmed'].astype(str).tolist()
print(f"Total dokumen: {len(texts)}")

# ------------------------------------------------------
# 2️ Model embedding + parameter clustering
# ------------------------------------------------------
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


umap_model = UMAP(
    n_neighbors=10,       # lebih kecil = cluster lebih detail
    n_components=5,
    min_dist=0.05,        # sedikit jarak agar tidak overmerge
    metric='cosine',
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=8,   # cluster minimum
    min_samples=3,        # untuk lebih robust terhadap noise
    metric='euclidean',
    cluster_selection_epsilon=0.05,
    cluster_selection_method='eom',
    prediction_data=True
)
# ------------------------------------------------------
# 3️ Inisialisasi BERTopic (banyak cluster dulu)
# ------------------------------------------------------
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="indonesian",
    min_topic_size=5,             # biarkan kecil untuk granular cluster
    nr_topics=None,               # None = biarkan auto dulu
    calculate_probabilities=True,
    representation_model=KeyBERTInspired(),
    verbose=True
)

# ------------------------------------------------------
# 4️ Fit ke data teks
# ------------------------------------------------------
topics, probs = topic_model.fit_transform(texts)
initial_num_topics = len(set(topics)) - (1 if -1 in topics else 0)
print(f"\nJumlah topik awal (belum direduksi): {initial_num_topics}")

# ------------------------------------------------------
# 5️ Evaluasi coherence & diversity
# ------------------------------------------------------
def evaluate_topics(model, docs):
    topic_words = [
        [word for word, _ in model.get_topic(topic)]
        for topic in model.get_topics().keys() if topic != -1
    ]
    tokenized_texts = [d.split() for d in docs]
    dictionary = Dictionary(tokenized_texts)

    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherence = coherence_model.get_coherence()

    unique_words = set([w for topic in topic_words for w in topic])
    diversity = len(unique_words) / (len(topic_words) * 10)

    return coherence, diversity

coh_before, div_before = evaluate_topics(topic_model, texts)
print(f"Coherence (awal): {coh_before:.4f}")
print(f"Diversity (awal): {div_before:.4f}")

# ------------------------------------------------------
# 6️ Reduksi topik ke bentuk lebih general (15–30)
# ------------------------------------------------------
target_topics = min(30, max(15, initial_num_topics // 2))  # adaptif
topic_model_reduced = topic_model.reduce_topics(texts, nr_topics=target_topics)
topics_reduced = topic_model_reduced.topics_

# Evaluasi setelah reduksi
coh_after, div_after = evaluate_topics(topic_model_reduced, texts)
print(f"\nSetelah Reduksi ke {target_topics} topik:")
print(f"Coherence: {coh_after:.4f}")
print(f"Diversity: {div_after:.4f}")

# ------------------------------------------------------
# 7️ Simpan hasil ke DataFrame
# ------------------------------------------------------
topic_info = topic_model_reduced.get_topic_info()
topic_info.to_csv("/content/topic_info_reduced.csv", index=False)
print("\nHasil topik tersimpan ke 'topic_info_reduced.csv'")

# ------------------------------------------------------
# 8️ (Opsional) Tampilkan 10 topik teratas
# ------------------------------------------------------
print("\n--- Top 10 Topik Teratas ---")
print(topic_info.head(10))


Total dokumen: 998


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


📊 Jumlah topik awal (belum direduksi): 19


2025-10-07 06:06:22,755 - BERTopic - Topic reduction - Reducing number of topics
2025-10-07 06:06:22,762 - BERTopic - Representation - Fine-tuning topics using representation models.


🧭 Coherence (awal): 0.3503
🌈 Diversity (awal): 0.7684


2025-10-07 06:06:23,945 - BERTopic - Representation - Completed ✓
2025-10-07 06:06:23,948 - BERTopic - Topic reduction - Reduced number of topics from 20 to 15



📉 Setelah Reduksi ke 15 topik:
🧭 Coherence: 0.4671
🌈 Diversity: 0.9714

✅ Hasil topik tersimpan ke 'topic_info_reduced.csv'

--- Top 10 Topik Teratas ---
   Topic  Count                                  Name  \
0     -1     98          -1_borneo_nama_tau_indonesia   
1      0    215    0_makeup_lipstick_mascara_skincare   
2      1    184        1_workout_workoutnya_gym_latih   
3      2    166           2_auto_motor_baterai_bensin   
4      3     56              3_hpnya_hp_laptop_tablet   
5      4     53   4_jakarta_indonesia_olimpiade_juang   
6      5     51  5_kamboja_cambodia_thailand_malaysia   
7      6     39    6_backhand_badminton_volley_tennis   
8      7     30             7_bekas_kompos_bikin_cuci   
9      8     25         8_ikanikan_ikan_fish_uburubur   

                                      Representation  \
0  [borneo, nama, tau, indonesia, laku, beli, bol...   
1  [makeup, lipstick, mascara, skincare, jerawat,...   
2  [workout, workoutnya, gym, latih, otot, dumbbe

In [ ]:
topic_info.to_csv('topic_info.csv',index=False)

In [ ]:
topic_model_reduced.visualize_topics()


In [ ]:
topic_model_reduced.visualize_hierarchy()


In [ ]:
topic_model_reduced.visualize_documents(texts)


In [ ]:
topic_model.visualize_documents(texts, hide_document_hover=False, custom_labels=True)


In [ ]:
# Melihat topik yang dihasilkan oleh model
topic_info = topic_model_reduced.get_topic_info()
topic_info.head(100)  # tampilkan topik-topik teratas


,Topic,Count,Name,Representation,Representative_Docs
0,-1,98,-1_borneo_nama_tau_indonesia,"[borneo, nama, tau, indonesia, laku, beli, bol...",[rivaldo nero pakpahan nama rivaldo nero pakpa...
1,0,215,0_makeup_lipstick_mascara_skincare,"[makeup, lipstick, mascara, skincare, jerawat,...",[ya temanteman siapsiap valentines day coba in...
2,1,184,1_workout_workoutnya_gym_latih,"[workout, workoutnya, gym, latih, otot, dumbbe...",[ya squat pegang dumbbell 812 kali dumbbell ro...
3,2,166,2_auto_motor_baterai_bensin,"[auto, motor, baterai, bensin, mesin, elektrik...",[mobil listrik ngechargengecharge pas kota nah...
4,3,56,3_hpnya_hp_laptop_tablet,"[hpnya, hp, laptop, tablet, prosesor, samsung,...",[hp 3 juta cakep ini kamera hp 3 juta bening m...
5,4,53,4_jakarta_indonesia_olimpiade_juang,"[jakarta, indonesia, olimpiade, juang, olahrag...",[assalamualaikum warahmatullahi wabarakatuh sa...
6,5,51,5_kamboja_cambodia_thailand_malaysia,"[kamboja, cambodia, thailand, malaysia, singap...",[tahan polisi daerah kamboja ya pas arah solat...
7,6,39,6_backhand_badminton_volley_tennis,"[backhand, badminton, volley, tennis, ball, la...",[halo teman cer suka main pucal pucal seru ya ...
8,7,30,7_bekas_kompos_bikin_cuci,"[bekas, kompos, bikin, cuci, pakai, kerudung, ...",[nurunin benda berat galon tas belanja apa mob...
9,8,25,8_ikanikan_ikan_fish_uburubur,"[ikanikan, ikan, fish, uburubur, koi, tonjol, ...",[koskosan ikan bulan 15 juta yep salah dengar ...


In [ ]:
# Buat kamus label manual
label_manual = {
    0: "Kecantikan (Skincare & Make-up)",
    1: "Olahraga (Kebugaran & Gym)",
    2: "Otomotif",
    3: "Barang Elektronik",
    4: "Olimpiade",
    5: "Traveling",
    6: "Olahraga Raket (Badminton, Tenis, Voli)",
    7: "Daur Ulang & Barang Bekas",
    8: "Perikanan",
    9: "Ucapan & Sosial Media",
    10: "Pendidikan",
    11: "Ekonomi",
    12:  "Islam & Religi",
    13: "Penyakit",
    -1: "Cluster_Lainnya"  # ini penting: untuk cluster abu-abu
}


In [ ]:
topic_info['Label_Manual'] = topic_info['Topic'].map(label_manual)


In [ ]:
transcribe['topic_id'] = topics
transcribe['topic_label'] = transcribe['topic_id'].map(label_manual)
transcribe.to_csv("/content/transcribe_with_topics.csv", index=False)
print("File tersimpan: transcribe_with_topics.csv")


✅ File tersimpan: transcribe_with_topics.csv


In [ ]:
transcribe[['base_name','text','topic_label']].to_csv('/content/transcribe_with_topics_text.csv',index=False)

In [ ]:
transcribe[['base_name','text','topic_label']].isnull().sum()

,0
base_name,0
text,2
topic_label,54


In [ ]:
print(sorted(transcribe['topic_id'].unique()))


[-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


In [ ]:
# --- Pastikan topik sesuai hasil reduksi ---
transcribe['topic_id'] = topics_reduced
transcribe['topic_id'] = transcribe['topic_id'].astype(int)

# --- Buat label manual (update jika ada topik baru) ---
label_manual = {
    0: "Kecantikan (Skincare & Make-up)",
    1: "Olahraga (Kebugaran & Gym)",
    2: "Otomotif",
    3: "Barang Elektronik",
    4: "Kompetisi Olahraga & Olimpiade",
    5: "Traveling",
    6: "Olahraga Raket (Badminton, Tenis, Voli)",
    7: "Daur Ulang & Barang Bekas",
    8: "Perikanan",
    9: "Ucapan & Sosial Media",
    10: "Pendidikan",
    11: "Ekonomi",
    12: "Islam & Religi",
    13: "Penyakit",
    -1: "Cluster_Lainnya"
}

# --- Mapping label ---
transcribe['topic_label'] = transcribe['topic_id'].map(label_manual)

# --- Cek topik yang belum punya label ---
print("Topik tanpa label:", set(transcribe['topic_id']) - set(label_manual.keys()))


Topik tanpa label: set()


In [ ]:
transcribe[['base_name','text','topic_label']].sample(10)

,base_name,text,topic_label
543,353,Lamborghini bilang ini adalah salah satu mobi...,Otomotif
887,71,"saya udah memberikan yang terbaik, udah go ab...",Cluster_Lainnya
291,111,sekarang nurunin benda berat kayak galon atau...,Daur Ulang & Barang Bekas
743,759,"Oke, Pak. Paling ini kita mencuri yang ini da...",Pendidikan
853,78,"Ya kalau kamu gak bawa balik, bodoh aja. Perl...",Traveling
563,385,"Hai, hari ini kita akan membahas lip product ...",Kecantikan (Skincare & Make-up)
140,649,"musik bismillahirrahmanirrahim bapak, mama ha...",Islam & Religi
823,952,kamu berada di dekat net dan ini menyerang la...,"Olahraga Raket (Badminton, Tenis, Voli)"
189,826,Kalau kalian belum pernah datang ke gym sama ...,Olahraga (Kebugaran & Gym)
343,676,"Saya pikir apa yang bisa kita lakukan adalah,...",Cluster_Lainnya


In [ ]:
transcribe.to_csv("/content/transcribe_with_topics_FIX.csv", index=False)
print("✅ File tersimpan: transcribe_with_topics.csv")

✅ File tersimpan: transcribe_with_topics.csv
